In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import rasterio
import streamlit as st

st.set_page_config(
    page_title="GeoVision | LULC Classification",
    page_icon="🛰️",
    layout="wide",
    initial_sidebar_state="expanded"
)

# The script sits in GeoVision/app; parent is the project folder.
PROJECT_ROOT = Path(__file__).resolve().parent.parent

outputs_maps_dir = PROJECT_ROOT / "outputs" / "maps"
outputs_reports_dir = PROJECT_ROOT / "outputs" / "reports"
models_dir = PROJECT_ROOT / "models"

MAP_PATH = outputs_maps_dir / "geovision_lulc_classified.tif"
METRICS_PATH = outputs_reports_dir / "model_metrics.json"
AREA_STATS_PATH = outputs_reports_dir / "land_cover_area_statistics.csv"
CONFUSION_MATRIX_PATH = outputs_reports_dir / "confusion_matrix.png"
REPORT_PATH = outputs_reports_dir / "classification_report.csv"
DISTRIBUTION_CHART_PATH = outputs_reports_dir / "land_cover_distribution.png"

CLASS_NAMES = {
    0: "Agriculture",
    1: "Built-up",
    2: "Forest",
    3: "Water",
    4: "Wasteland",
    5: "Roads"
}

CLASS_COLORS = {
    0: [230, 210, 70],
    1: [230, 60, 60],
    2: [30, 140, 50],
    3: [40, 110, 230],
    4: [170, 120, 70],
    5: [190, 190, 190]
}


@st.cache_data
def load_classification_map(path_string):
    with rasterio.open(path_string) as src:
        return src.read(1), src.crs, src.transform


@st.cache_data
def make_rgb_classification(classification):
    h, w = classification.shape
    image = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id, colour in CLASS_COLORS.items():
        image[classification == class_id] = colour

    return image


@st.cache_data
def load_table(path_string):
    return pd.read_csv(path_string)


def make_legend():
    return [
        Patch(
            facecolor=np.array(CLASS_COLORS[class_id]) / 255,
            label=CLASS_NAMES[class_id]
        )
        for class_id in CLASS_NAMES
    ]


# Sidebar
st.sidebar.title("🛰️ GeoVision")
st.sidebar.caption("AI-based Land Use / Land Cover Classification")

st.sidebar.markdown("---")
st.sidebar.subheader("Study-area settings")

location = st.sidebar.text_input(
    "Selected area",
    value="Sentinel-2 study area, India"
)

start_date = st.sidebar.date_input(
    "Image start date",
    value=None,
    disabled=True
)

st.sidebar.info(
    "Prototype mode: this dashboard displays a processed Sentinel-2 "
    "Level-2A study area. The next version will allow users to draw any "
    "area in India and classify it dynamically."
)

st.sidebar.markdown("---")
st.sidebar.subheader("Layer controls")

show_agriculture = st.sidebar.checkbox("Agriculture", value=True)
show_builtup = st.sidebar.checkbox("Built-up", value=True)
show_forest = st.sidebar.checkbox("Forest", value=True)
show_water = st.sidebar.checkbox("Water", value=True)
show_wasteland = st.sidebar.checkbox("Wasteland", value=True)
show_roads = st.sidebar.checkbox("Roads", value=True)

selected_layers = {
    0: show_agriculture,
    1: show_builtup,
    2: show_forest,
    3: show_water,
    4: show_wasteland,
    5: show_roads
}

# Main page
st.title("GeoVision")
st.subheader("AI-Based Land Use and Land Cover Classification System")

st.write(
    "GeoVision processes Sentinel-2 Level-2A multispectral imagery using "
    "spectral indices and a Random Forest classifier. It classifies land "
    "into agriculture, built-up areas, forest, water bodies, wasteland, and roads."
)

if not MAP_PATH.exists():
    st.error(
        "Classification map not found. Run the notebook export cell first so that "
        "`outputs/maps/geovision_lulc_classified.tif` exists."
    )
    st.stop()

if not AREA_STATS_PATH.exists():
    st.error(
        "Area statistics are missing. Run the final map export cell in the notebook."
    )
    st.stop()

classification, raster_crs, raster_transform = load_classification_map(str(MAP_PATH))
classification_rgb = make_rgb_classification(classification)

# Apply sidebar layer visibility
display_image = classification_rgb.copy()

for class_id, selected in selected_layers.items():
    if not selected:
        display_image[classification == class_id] = [245, 245, 245]

area_df = load_table(str(AREA_STATS_PATH))

# Summary tiles
total_area_km2 = area_df["area_km2"].sum()
dominant_row = area_df.loc[area_df["percentage"].idxmax()]
classified_area_ha = area_df["area_hectares"].sum()

metric_col1, metric_col2, metric_col3, metric_col4 = st.columns(4)

metric_col1.metric("Study-area size", f"{total_area_km2:.2f} km²")
metric_col2.metric("Classified area", f"{classified_area_ha:,.0f} ha")
metric_col3.metric("Dominant land cover", dominant_row["class_name"])
metric_col4.metric("Dominant share", f"{dominant_row['percentage']:.2f}%")

st.markdown("---")

map_col, stats_col = st.columns([1.6, 1])

with map_col:
    st.header("Classified LULC Map")
    st.image(
        display_image,
        caption=f"Selected study area: {location}",
        use_container_width=True
    )

    st.caption(
        "Colour-coded classes: Agriculture (yellow), Built-up (red), Forest (green), "
        "Water (blue), Wasteland (brown), Roads (grey)."
    )

with stats_col:
    st.header("Land-Cover Statistics")

    display_table = area_df[
        ["class_name", "area_hectares", "area_km2", "percentage"]
    ].copy()

    display_table.columns = [
        "Class",
        "Area (ha)",
        "Area (km²)",
        "Share (%)"
    ]

    st.dataframe(
        display_table,
        use_container_width=True,
        hide_index=True
    )

    st.bar_chart(
        area_df.set_index("class_name")["percentage"],
        color="#3B82F6"
    )

st.markdown("---")
st.header("Class Legend")

legend_cols = st.columns(6)

for column, class_id in zip(legend_cols, CLASS_NAMES):
    colour = CLASS_COLORS[class_id]
    hex_colour = "#{:02x}{:02x}{:02x}".format(*colour)

    column.markdown(
        f"""
        <div style="
            background-color:{hex_colour};
            color:#111111;
            padding:14px 8px;
            border-radius:8px;
            border:1px solid #D1D5DB;
            text-align:center;
            font-weight:700;
            min-height:52px;">
            {CLASS_NAMES[class_id]}
        </div>
        """,
        unsafe_allow_html=True
    )

st.markdown("---")
st.header("Model Evaluation")

if METRICS_PATH.exists():
    with open(METRICS_PATH, "r", encoding="utf-8") as file:
        metrics = json.load(file)

    evaluation_col1, evaluation_col2, evaluation_col3 = st.columns(3)

    evaluation_col1.metric(
        "Overall Accuracy",
        f"{float(metrics.get('overall_accuracy', 0)) * 100:.2f}%"
    )
    evaluation_col2.metric(
        "Kappa Score",
        f"{float(metrics.get('kappa_score', 0)):.3f}"
    )
    evaluation_col3.metric(
        "Classifier",
        metrics.get("model", "Random Forest")
    )

    st.warning(
        metrics.get(
            "training_note",
            "Prototype evaluation results should be interpreted with the training data scope."
        )
    )

if CONFUSION_MATRIX_PATH.exists() and REPORT_PATH.exists():
    evaluation_map_col, evaluation_table_col = st.columns([1, 1])

    with evaluation_map_col:
        st.image(
            str(CONFUSION_MATRIX_PATH),
            caption="Confusion Matrix",
            use_container_width=True
        )

    with evaluation_table_col:
        report_df = load_table(str(REPORT_PATH))
        st.dataframe(
            report_df,
            use_container_width=True,
            hide_index=True
        )
else:
    st.info(
        "Evaluation images/tables will appear after running the metrics-export notebook cells."
    )

st.markdown("---")
st.header("Applications")

application_col1, application_col2, application_col3 = st.columns(3)

with application_col1:
    st.subheader("Urban Planning")
    st.write(
        "Detect built-up expansion, identify land-conversion patterns, and support "
        "infrastructure and zoning decisions."
    )

with application_col2:
    st.subheader("Agriculture Monitoring")
    st.write(
        "Estimate agricultural land extent, observe vegetation patterns, and support "
        "crop and irrigation monitoring."
    )

with application_col3:
    st.subheader("Environmental Analysis")
    st.write(
        "Monitor tree cover, water bodies, wasteland change, and other indicators "
        "useful for conservation planning."
    )

st.caption(
    "GeoVision prototype | Sentinel-2 Level-2A imagery | Random Forest classification | "
    "10 m spatial resolution"
)

ModuleNotFoundError: No module named 'streamlit'